# Data Inspection
Loads a TWIX scan, extracts k-space, zero-fills it, and displays a preview as an animated GIF. This helps you determine the number of phase-encode lines collected and the offset. Then you can update those values in your configuration file accordingly.

### Loading packages and data

In [1]:
import yaml
import numpy as np
import utils.data_ingestion as di
import utils.gif as gif
from IPython.display import display

def load_config(config_file="config.yaml"):
    """
    Load configuration from a YAML file.

    Parameters
    ----------
    config_file : str
        Path to the YAML configuration file.

    Returns
    -------
    dict
        Parsed configuration data.
    """
    with open(config_file, "r") as f:
        config = yaml.safe_load(f)
    return config

# Read configuration
config = load_config()

# Extract paths and other parameters from config
twix_file = config["data"]["twix_file"]
dicom_folder = config["data"]["dicom_folder"]

# Read TWIX file (only the last scan by default in this example)
scans = di.read_twix_file(twix_file, include_scans=[-1], parse_pmu=False)

Software version: VD/VE (!?)

Scan  1


100%|██████████| 287M/287M [00:00<00:00, 1.44GB/s]

Read 1 scans from 20250307_JM/raw/meas_MID00115_FID27380_Cor_250306_2.dat


### Extracting k-space data, zero-filling, and displaying

In [2]:
# Extract raw k-space data
kspace = di.extract_image_data(scans[-1], full_kspace_shape=(100, 128, 30, 256), ref_scan=True)
# kspace = di.extract_image_data(scans[-1], ref_scan=True)
# kspace = kspace.reshape((150, -1, 20, 256))

# Display k-space as an animated GIF
preview = gif.display_kspace_as_gif(kspace, duration=0.2)
display(preview)

Extracted image data shape: (100, 128, 30, 256)


In [32]:
from utils.reconstruction import grappa_reconstruction, direct_ifft_reconstruction, tgrappa_reconstruction

# Figure out which lines are acquired
acquired_lines = np.where(~np.all(kspace == 0, axis=(0, 2, 3)))[0]
# Example: acquired_lines = [0, 2, 4, 6, 7, 8, 9, 10, 12, 14, 16]

# Figure out the largest contiguous block of acquired lines
contiguous_lines = np.split(acquired_lines, np.where(np.diff(acquired_lines) != 1)[0] + 1)
largest_block = max(contiguous_lines, key=len)
smallest_line, largest_line = largest_block[0], largest_block[-1]
print(f"Smallest line: {smallest_line}, Largest line: {largest_line}")

print("Performing GRAPPA...")
# images = grappa_reconstruction(kspace[:10, ...], calib_region=(smallest_line, largest_line), kernel_size=(3, 5))
images = tgrappa_reconstruction(kspace[:10, ...], calib_size=(20, 20), kernel_size=(5, 5))
# images = direct_ifft_reconstruction(kspace, use_conjugate_symmetry=True)
print("Reconstruction complete.")

Smallest line: 36, Largest line: 60
Performing GRAPPA...


Reconstruction complete.


In [33]:
# edited_images = np.rot90(images, k=1, axes=(1, 2))
# edited_images = np.flip(edited_images, axis=2)
# edited_images = edited_images[:, 64:-64, :]
# edited_images = np.rot90(edited_images, k=1, axes=(1, 2))
edited_images = np.flip(images, axis=1)
edited_images = edited_images[:, :, 64:-64]
preview = gif.display_images_as_gif(edited_images, notebook=True)
display(preview)

In [4]:
# # Extract raw k-space data
# kspace = di.extract_image_data(scans[-1])

# n_frames = di.get_num_frames(dicom_folder)
# n_coils = kspace.shape[1]

# # Reshape k-space into frames
# kspace = np.reshape(kspace, (n_frames, -1, n_coils, kspace.shape[2]))

# # Pull the total number of phase encodes and define offset
# extended_pe_lines = di.get_total_phase_encodes(dicom_folder)
# offset = 32  # Adjust if needed

# # Allocate zero-filled array
# kspace_zf = np.zeros((n_frames, extended_pe_lines, n_coils, kspace.shape[3]), dtype=np.complex64)
# kspace_zf[:, offset : offset + kspace.shape[1], :] = kspace

# # Display k-space as an animated GIF
# preview = gif.display_kspace_as_gif(kspace_zf, duration=0.2)
# display(preview)

In [6]:
# from utils.reconstruction import direct_ifft_reconstruction
# images = direct_ifft_reconstruction(kspace, extended_pe_lines, 0, use_conjugate_symmetry=False)
# images = np.rot90(images, k=1, axes=(1, 2))
# images = np.flip(images, axis=2)
# images = images[:, 64:-64, :]
# preview = gif.display_images_as_gif(images, notebook=True)
# display(preview)

In [3]:
for (i,mdb) in enumerate(scans[-1]['mdb']):
    if mdb.is_image_scan():
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
        )
    elif mdb.is_flag_set('PATREFSCAN'):
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
            "REFERENCE"
        )

0 0 0
1 0 0
2 0 0
3 0 0
4 0 0
5 0 0
6 0 0
7 0 0
8 0 0
9 0 0
10 0 0
11 0 0
12 0 0
13 0 0
14 0 0
15 0 0
16 0 0
17 0 0
18 0 0
19 0 0
20 0 0
21 0 0
22 0 0
23 0 0
24 0 0
25 0 0
26 0 0
27 0 0
28 0 0
29 0 0
30 0 0
31 0 0
32 0 0
33 0 0
34 0 0
35 0 0
36 0 0
37 0 0
38 0 0
39 0 0
40 0 0
41 0 0
42 0 0
43 0 0
44 0 0
45 0 0
46 0 0
47 0 0
0 1 0
1 1 0
2 1 0
3 1 0
4 1 0
5 1 0
6 1 0
7 1 0
8 1 0
9 1 0
10 1 0
11 1 0
12 1 0
13 1 0
14 1 0
15 1 0
16 1 0
17 1 0
18 1 0
19 1 0
20 1 0
21 1 0
22 1 0
23 1 0
24 1 0
25 1 0
26 1 0
27 1 0
28 1 0
29 1 0
30 1 0
31 1 0
32 1 0
33 1 0
34 1 0
35 1 0
36 1 0
37 1 0
38 1 0
39 1 0
40 1 0
41 1 0
42 1 0
43 1 0
44 1 0
45 1 0
46 1 0
47 1 0
0 2 0
1 2 0
2 2 0
3 2 0
4 2 0
5 2 0
6 2 0
7 2 0
8 2 0
9 2 0
10 2 0
11 2 0
12 2 0
13 2 0
14 2 0
15 2 0
16 2 0
17 2 0
18 2 0
19 2 0
20 2 0
21 2 0
22 2 0
23 2 0
24 2 0
25 2 0
26 2 0
27 2 0
28 2 0
29 2 0
30 2 0
31 2 0
32 2 0
33 2 0
34 2 0
35 2 0
36 2 0
37 2 0
38 2 0
39 2 0
40 2 0
41 2 0
42 2 0
43 2 0
44 2 0
45 2 0
46 2 0
47 2 0
0 3 0
1 3 0
2 3 0
3 3 